In [ ]:
# Installing necessary packages to run future code

%pip install pandas numpy scikit-learn matplotlib
%pip install openpyxl

In [ ]:
# Organising environmental sensor data so that it is in a nice format

import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")  # put your three files in a "data" folder next to this script

# 1. Load files (default C engine; no rows are silently skipped)
environmental_df = pd.read_csv(DATA_DIR / "Environmental sensor.csv")

# Ensure the columns index and row index are flat and clean
environmental_df.columns = [str(col) for col in environmental_df.columns]
environmental_df.reset_index(drop=True, inplace=True)

# 2. Define the JSON parsing helper function
def parse_sensor_json(json_str):
    if pd.isna(json_str):
        return {}
    try:
        if isinstance(json_str, str):
            if json_str.startswith('""') or '""' in json_str:
                json_str = json_str.replace('""', '"')
            if json_str.startswith('"') and json_str.endswith('"'):
                json_str = json_str[1:-1]
            data = json.loads(json_str)
        else:
            data = json_str
            
        extracted = {}
        if isinstance(data, list):
            for item in data:
                var_info = item.get('variable', {})
                var_name = var_info.get('name')
                values_list = item.get('values', [])
                if values_list and isinstance(values_list, list):
                    metric_value = values_list[0].get('value')
                    if var_name is not None:
                        extracted[var_name] = metric_value
        return extracted
    except Exception:
        return {}

# 3. Parse and extract key-value metrics from environmental data
parsed_metrics = environmental_df['jsondata'].apply(parse_sensor_json)
metrics_df = pd.DataFrame(parsed_metrics.tolist())
metrics_df.reset_index(drop=True, inplace=True)

# Map standard headings to include units explicitly
unit_mapping = {
    'Battery': 'Battery (V)',
    'Carbon dioxide': 'Carbon dioxide (ppm)',
    'Humidity': 'Humidity (%)',
    'Formaldehyde': 'Formaldehyde (mg/m³)',
    'Pressure': 'Pressure (mb)',
    'Temperature': 'Temperature (°C)',
    'Light': 'Light (lx)',
    'Light Volatile Organic Compounds': 'Light Volatile Organic Compounds (ppb)',
    'Particulate matter 1': 'Particulate matter 1 (µg/m³)',
    'Particulate matter 4': 'Particulate matter 4 (µg/m³)',
    'Particulate matter 2.5': 'Particulate matter 2.5 (µg/m³)',
    'Particulate matter 10': 'Particulate matter 10 (µg/m³)'
}
metrics_df.rename(columns=unit_mapping, inplace=True)

# 4. Combine extracted metrics with the original DataFrame (dropping original raw JSON column)
environmental_expanded_df = pd.concat(
    [environmental_df.drop(columns=['jsondata']), metrics_df],
    axis=1
)

# 5. Display summaries and verify columns directly
print("--- Verified Column Names in Environmental Data ---")
for col in environmental_expanded_df.columns:
    print(f" - {col}")

print("\n--- Expanded Environmental Sensor Data with Units ---")
display(environmental_expanded_df.head())

